# 11 · Final model, one-time tests, and the scores Salud receives

1. **Choose before looking at any test set** — from the cross-validation leaderboard only (notebooks 02–08).
2. **Calibration:** none vs Platt vs isotonic, by cross-validation.
3. **One-time synthetic holdout test** (20% of patients never used so far).
4. **One-time real test** (Salud's 58 usable rows): frozen model, bootstrap CIs, predicted vs actual, recalibrating
   the overall level (leave-one-out), and the training-source comparison (DS-03).
5. **Score every real patient** with an open balance as of the data's end date → tiers, expected dollars, account list,
   fairness check → `reports/phase2/propensity_scores.xlsx`.

**Important — the real test is no longer independent.** The synthetic world (`WORLD = "salud"`) was tuned so its case
mix, distributions and each feature's direction of effect match these same 84 real calls (`salud-synth-calibrate`).
A good real-data result therefore shows the model is *consistent* with Salud's data, not that it generalises to new
patients. An independent check needs new real data from Salud.

*Privacy: real data is shown only as aggregates; the scores file is written to `reports/` (gitignored).*

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "11"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
F = M.FS_TRANSFER
MODELS = {
    "Logistic regression (L2)": lambda: M.pipeline(M.preprocess("z"), M.logreg(C=0.1)),
    "Scorecard (monotone WoE)": lambda: M.pipeline([("woe", M.WoE(monotonic=True))], M.logreg(C=1.0)),
    "LightGBM (monotone)": lambda: M.pipeline(M.preprocess("none"), M.lgbm(F, monotone=True, num_leaves=7, learning_rate=.03,
                                     n_estimators=300, min_child_samples=20, subsample=.8, subsample_freq=1, colsample_bytree=.8)),
    "Extra trees": lambda: M.pipeline(M.preprocess("none"), ExtraTreesClassifier(400, min_samples_leaf=10, max_features=.5, random_state=M.SEED, n_jobs=-1)),
    "Neural network": lambda: M.pipeline(M.preprocess("log_z"), MLPClassifier((16,), alpha=1.0, early_stopping=True, max_iter=2000, random_state=M.SEED)),
}

## 1. The choice (cross-validation only)

In [ ]:
FINALISTS = {"SC-05": ("Scorecard (monotone WoE)", lambda: M.pipeline(M.preprocess("woe_mono"), M.IVFilter(.02), M.logreg(C=1.0))),
             "LR-12": ("Logistic regression (L2)", lambda: M.tuned(M.pipeline(M.preprocess("z"), M.logreg(C=1.0)), {"logisticregression__C": loguniform(1e-3, 1e2)})),
             "GB-06": ("LightGBM (monotone)", MODELS["LightGBM (monotone)"])}
fin = M.leaderboard().query("run_id in @FINALISTS").reset_index(drop=True)
CHOSEN = M.one_se_pick(fin).run_id   # one-standard-error rule on cross-validation only; fixed BEFORE any test below
print("Chosen (simplest within one standard error of the best, cross-validation only):", CHOSEN)
fin.set_index("run_id")[["description", "auc", "auc_se", "brier", "cal_slope", "n_features"]]

## 2. Calibration (cross-validated, the same 25 folds)

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
for rid, (name, mk) in FINALISTS.items():
    for method in ("sigmoid", "isotonic"):
        run(f"CAL-{rid}-{method[:3]}", CalibratedClassifierCV(mk(), method=method, cv=5), F, f"{name} + {method} calibration")
cal = M.leaderboard(notebook=NB)
base = M.leaderboard().set_index("run_id").loc[list(FINALISTS)].reset_index()[cal.columns]
pd.concat([base, cal])[["run_id", "description", "auc", "brier", "log_loss", "cal_slope"]].round(3)

## 3. One-time synthetic holdout test

In [ ]:
Xd, yd = M.xy(dev, F); Xh, yh = M.xy(holdout, F)
fitted = {rid: mk().fit(Xd, yd) for rid, (name, mk) in FINALISTS.items()}
rows = []
for rid, m in fitted.items():
    p = m.predict_proba(Xh)[:, 1]; est, lo, hi = M.bootstrap_ci(yh, p)
    rows.append({"run": rid, "model": FINALISTS[rid][0], "AUC": est, "95% CI": f"{lo:.3f}–{hi:.3f}", **{k: v for k, v in M.metrics(yh, p).items() if k != "auc"}})
pd.DataFrame(rows).round(3)

## 4. One-time real test (58 rows from one day, Jul 1 2026)
The models were trained only on synthetic data and are frozen. With 58 rows the AUC's 95% interval is about ±0.15:
read the result as an indication, not proof.

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
Xr, yr = M.xy(real, F)
rows = []
for rid, m in fitted.items():
    p = m.predict_proba(Xr)[:, 1]; est, lo, hi = M.bootstrap_ci(yr, p)
    rows.append({"run": rid, "model": FINALISTS[rid][0], "AUC": est, "95% CI": f"{lo:.3f}–{hi:.3f}", "Brier": brier_score_loss(yr, p),
                 "mean predicted": p.mean(), "actual pay rate": yr.mean()})
rule = (Xr.n_payments_365d > 0) * 1000 - Xr.days_since_last_payment.fillna(999)
e, lo, hi = M.bootstrap_ci(yr, rule.to_numpy(float))
rows.append({"run": "B-03", "model": "Rule: paid before, most recent first", "AUC": e, "95% CI": f"{lo:.3f}–{hi:.3f}"})
real_test = pd.DataFrame(rows); real_test.round(3)

### Recalibrating the overall level for Salud (leave-one-out)
Synthetic patients pay at a different rate than Salud's. A standard fix when moving a model to new data is to keep
its ranking and shift only its intercept to the new pay rate. Leave-one-out: each real row is scored by a shift
fitted on the other 57.

In [ ]:
from scipy.special import logit, expit
from scipy.optimize import brentq
def shift_for(p, y):   # intercept shift that makes the mean predicted probability equal the observed rate
    return brentq(lambda d: expit(logit(np.clip(p, 1e-6, 1 - 1e-6)) + d).mean() - y.mean(), -10, 10)
rows = []
for rid, m in fitted.items():
    p = np.clip(m.predict_proba(Xr)[:, 1], 1e-6, 1 - 1e-6)
    loo = np.array([expit(logit(p[i]) + shift_for(np.delete(p, i), np.delete(yr, i))) for i in range(len(yr))])
    rows.append({"run": rid, "Brier frozen": brier_score_loss(yr, p), "Brier recalibrated (LOO)": brier_score_loss(yr, loo),
                 "shift (log-odds)": shift_for(p, yr), "AUC (unchanged)": roc_auc_score(yr, p)})
pd.DataFrame(rows).round(3)

### DS-03 · Training source, tested on the real rows
(a) synthetic only, frozen · (b) real only — a 2-feature logistic regression, leave-one-out (features fixed in advance:
payments in the last year, days since the newest placement) · (c) synthetic model + real intercept shift (LOO, above).

In [ ]:
from sklearn.model_selection import LeaveOneOut
two = ["n_payments_365d", "days_since_last_placement"]
Xr2 = M.xy(real, two)[0]; p_real = np.zeros(len(yr))
for tr, te in LeaveOneOut().split(Xr2):
    p_real[te] = M.pipeline(M.preprocess("log_z"), M.logreg(C=1.0)).fit(Xr2.iloc[tr], yr[tr]).predict_proba(Xr2.iloc[te])[:, 1]
e, lo, hi = M.bootstrap_ci(yr, p_real)
p_syn = fitted[CHOSEN].predict_proba(Xr)[:, 1]; e2, lo2, hi2 = M.bootstrap_ci(yr, p_syn)
pd.DataFrame([{"source": f"(a) synthetic only, frozen ({CHOSEN})", "AUC": e2, "95% CI": f"{lo2:.3f}–{hi2:.3f}", "Brier": brier_score_loss(yr, p_syn)},
              {"source": "(b) real only, 2 features, leave-one-out", "AUC": e, "95% CI": f"{lo:.3f}–{hi:.3f}", "Brier": brier_score_loss(yr, p_real)}]).round(3)

### Pay rate by score group on the real rows (chosen model, three equal groups of ~19)

In [ ]:
grp = pd.qcut(pd.Series(p_syn).rank(method="first"), 3, labels=["lowest third", "middle third", "highest third"])
pd.DataFrame({"group": grp, "paid": yr}).groupby("group", observed=True).paid.agg(["size", "mean"]).rename(columns={"size": "patients", "mean": "pay rate"}).round(2)

### Post-call model on the real calls (no payment taken on the call; synthetic-trained, no QA score)

In [ ]:
call_dev, _, real_all = M.load_data(design="call", world=WORLD)
post_tr = M.post_rows(call_dev); post_real = M.post_rows(real)
SENT = ["caller_sentiment_overall", "caller_sentiment_end", "caller_sentiment_shift", "agent_sentiment_overall"]
OUTC = ["inbound", "spoke_with_agent", "collection_attempted", "promised_to_pay", "non_committal", "contested_or_refused",
        "pursued_insurance", "transferred", "action_shared_balance", "action_payment_plan", "action_financial_assistance",
        "action_statement_or_bill", "action_insurance", "n_actions", "asked_financial_assistance", "talk_seconds"]
rows = []
for label, feats in [("pre-call features", F), ("+ call outcome + sentiment", F + OUTC + SENT)]:
    mdl = M.pipeline(M.preprocess("log_z"), M.logreg(C=0.1)).fit(*M.xy(post_tr, feats, M.TARGET_POST))
    Xp, yp = M.xy(post_real, feats, M.TARGET_POST); p = mdl.predict_proba(Xp)[:, 1]
    e, lo, hi = M.bootstrap_ci(yp, p)
    rows.append({"features": label, "real rows": len(yp), "paid after call": int(yp.sum()), "AUC": e, "95% CI": f"{lo:.3f}–{hi:.3f}"})
pd.DataFrame(rows).round(3)

## 5. Score every real patient (as of the data's end date) and write the deliverable
The chosen model is refitted on **all** synthetic fixed-date rows (development + holdout), then its overall level is
shifted to Salud's observed pay rate (section 4). Tiers are fifths of the ranked list; each has an action consistent
with fair, non-pressuring collections.

In [ ]:
from salud_pipeline.phase2.propensity.sources import load_real
from salud_pipeline.phase2.propensity.snapshots import build, fixed_date_world, open_balance
RW = load_real()
as_of = RW.data_end.normalize()
now = open_balance(build(fixed_date_world(RW, as_of)))
for b in now.select_dtypes("bool"): now[b] = now[b].astype(int)
full = pd.concat([dev, holdout], ignore_index=True)
final = FINALISTS[CHOSEN][1]().fit(*M.xy(full, F))
shift = shift_for(np.clip(final.predict_proba(Xr)[:, 1], 1e-6, 1 - 1e-6), yr)
now["propensity"] = expit(logit(np.clip(final.predict_proba(M.xy(now, F)[0])[:, 1], 1e-6, 1 - 1e-6)) + shift)
now["expected_30d_dollars"] = now.propensity * now.balance_at_t
now["rank"] = now.propensity.rank(ascending=False, method="first").astype(int)
ACTIONS = {1: "Light reminder or self-serve payment link (likely to pay without a call)",
           2: "Priority agent call", 3: "Agent call",
           4: "Agent call offering a payment plan",
           5: "Financial-assistance screening and payment-plan offer (no pressure)"}
now["tier"] = pd.qcut(now["rank"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
now["suggested_action"] = now.tier.map(ACTIONS)
lang = RW.calls.dropna(subset=["guarantor_id"]).sort_values("t").groupby("guarantor_id").language.last()
now["preferred_language"] = now.guarantor_id.map(lang)
print(f"Scored {len(now)} patients with an open balance as of {as_of:%Y-%m-%d}; recalibration shift {shift:+.2f} log-odds")
now.groupby("tier").agg(patients=("guarantor_id", "size"), mean_propensity=("propensity", "mean"),
                        total_balance=("balance_at_t", "sum"), expected_30d_dollars=("expected_30d_dollars", "sum")).round(2)

In [ ]:
fair = now.groupby(now.preferred_language.fillna("unknown")).agg(patients=("guarantor_id", "size"), mean_propensity=("propensity", "mean"),
                                                                 share_in_tier_1_2=("tier", lambda t: (t <= 2).mean()))
fair.round(3)

In [ ]:
# Account list: each account inherits its patient's score (accounts placed on or before the scoring date)
acc = RW.accounts[RW.accounts.placement_date <= as_of].merge(now[["guarantor_id", "propensity", "tier", "rank", "suggested_action"]], on="guarantor_id")
# Factor weights: the scorecard's points per bin, and the coefficients of the transferable logistic model
X_full, y_full = M.xy(full, F)
sc_full = FINALISTS["SC-05"][1]().fit(X_full, y_full)   # the points table always comes from the scorecard
woe, ivf, lr = sc_full[0], sc_full[1], sc_full[-1]
factor = 20 / np.log(2); nfeat = len(ivf.keep_); card = []
for f_, beta in zip(ivf.keep_, lr.coef_[0]):
    edges = woe.edges_[f_]
    for b, w in sorted(woe.table_[f_].items()):
        if b == -1: lab = "missing / never"
        else:
            lo_ = -np.inf if b == 0 else edges[int(b) - 1]; hi_ = np.inf if b == len(edges) else edges[int(b)]
            lab = f"[{lo_:,.1f}, {hi_:,.1f})"
        card.append({"feature": f_, "bin": lab, "WoE": round(w, 3), "points": int(round(factor * (beta * w + lr.intercept_[0] / nfeat) + 600 / nfeat))})
card = pd.DataFrame(card)
rt = real_test.set_index("run")
auc_c, ci_c = rt.loc[CHOSEN, "AUC"], rt.loc[CHOSEN, "95% CI"]
ci_low = float(ci_c.split("–")[0])
REAL_VERDICT = (f"{'Better than chance' if ci_low > .5 else 'NOT better than chance'} on Salud's 58 real rows: AUC {auc_c:.2f} "
                f"(95% CI {ci_c}); rule 'paid before, most recent first' {rt.loc['B-03', 'AUC']:.2f}. Not an independent test: "
                f"the synthetic world ({WORLD}) was tuned to these calls' aggregate statistics.")
assumptions = pd.DataFrame([
    ("Model", f"{CHOSEN}: {FINALISTS[CHOSEN][0]} on FS_TRANSFER, trained on synthetic fixed-date rows (dev + holdout) of the '{WORLD}' world"),
    ("Scoring date", f"{as_of:%Y-%m-%d} (last transaction date in Salud's package)"),
    ("Who is scored", "Patients with at least one placed account and a balance left (rebuilt from placements minus patient payments)"),
    ("Outcome", "Any patient payment within 30 days"),
    ("Recalibration", f"Intercept shifted {shift:+.2f} log-odds so the average matches Salud's observed 30-day pay rate (58 real rows)"),
    ("Tiers", "Fifths of the ranked list (1 = most likely to pay)"),
    ("Cut-off for calling", "Not set: needs Salud's cost per call and recovery value; call when propensity × balance > cost of a call"),
    ("Language", "Never a model input; shown only for the fairness check"),
    ("Evidence", "Real-data AUC is from 58 rows on one day — an indication, not proof (see Model comparison)"),
    ("Real-data test RESULT", REAL_VERDICT),
], columns=["item", "value"])
warning = pd.DataFrame({"READ FIRST": ([
    "INDICATIVE SCORES — NOT YET VALIDATED ON NEW DATA.",
    f"The model ranks Salud's real rows better than chance (AUC {auc_c:.2f}, 95% CI {ci_c}), but the synthetic training world was tuned to these same calls.",
    "Use for piloting and review with Salud, not for automated decisions, until tested on new real data.",
] + ([f"The simple rule 'paid before, most recent payment first' scores {rt.loc['B-03', 'AUC']:.2f} on the same rows, "
      f"so it is at least as good: use it as the call order and show this score beside it."]
     if rt.loc['B-03', 'AUC'] >= auc_c else []) if ci_low > .5 else [
    "DO NOT USE THESE SCORES FOR DECISIONS.",
    f"The model did not rank Salud's real rows better than chance (AUC {auc_c:.2f}, 95% CI {ci_c}).",
]) + ["This file shows the end-to-end output format (ranking, tiers, actions, expected dollars, account list, scorecard)."]})
out = M.RESULTS.parent.parent / "propensity_scores.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as xw:
    warning.to_excel(xw, sheet_name="READ FIRST", index=False)
    now.sort_values("rank")[["rank", "guarantor_id", "tier", "suggested_action", "propensity", "balance_at_t", "expected_30d_dollars",
                             "n_accounts", "n_payments_365d", "days_since_last_payment", "days_since_last_placement"]].to_excel(xw, sheet_name="Patient ranking", index=False)
    acc.sort_values(["rank", "placement_date"])[["rank", "guarantor_id", "account_id", "tier", "propensity", "hbpb", "placement_date",
                                                 "placement_balance", "suggested_action"]].to_excel(xw, sheet_name="Account list", index=False)
    M.leaderboard().to_excel(xw, sheet_name="Model comparison", index=False)
    card.to_excel(xw, sheet_name="Scorecard points", index=False)
    real_test.to_excel(xw, sheet_name="Real-data test", index=False)
    assumptions.to_excel(xw, sheet_name="Assumptions", index=False)
print(f"Wrote {out}: {len(now)} patients, {len(acc)} accounts")

## Findings (Salud-tuned world)
- **Choice (cross-validation only):** **GB-06**, monotone LightGBM on FS_TRANSFER, AUC 0.876. It is more than one SE above SC-05 (0.871) and LR-12 (0.866).
  Sigmoid calibration fixes its under-confidence (Brier 0.132 → 0.119).
- **Synthetic holdout (latest 20%, used once):** GB-06 0.829 (0.78–0.88), SC-05 0.814, LR-12 0.802. That is about 0.05 below cross-validation, a drift over time inside the world.
- **Real test (58 rows):**

  | Model | AUC (95% CI) |
  |---|---|
  | GB-06 | **0.697** (0.55–0.84) |
  | SC-05 | 0.631 |
  | LR-12 | 0.633 |
  | Rule B-03 | 0.718 (0.61–0.83) |

  The original world's scorecard reached 0.448.
  Pay rate by score third: **35% → 42% → 74%**. The ranking now points the right way, but it does **not beat the simple rule**.
- **Level:** the models predict about 0.25 against an actual 0.50. A leave-one-out intercept shift (+1.7 log-odds for GB-06) fixes this: Brier 0.287 → 0.219.
- **Not an independent test:** the world was tuned to these same 84 calls, so 0.697 measures consistency, not generalisation.
  An honest test needs new Salud calls.
- **Deliverable:** `reports/phase2/propensity_scores.xlsx`, covering 46 patients with an open balance, in five tiers. Tier 1 has a mean propensity of 0.95 and holds $62k of expected 30-day dollars.
  Recommendation: use the rule "paid before, most recent first" as the baseline policy until more real data arrives. Show the model score next to it.